# Lab 01 — Leakage-safe intrusion detection with UNSW-NB15

This notebook implements the complete requested pipeline: audit, cleaning, train-only preprocessing, validation-based model selection, multiclass and binary evaluation, and an imbalance experiment. **Run cells in order.** The supplied testing CSV is deliberately held back until the final scoring cells.

## 0. Setup and reproducibility

Standard Google Colab includes these libraries. `RANDOM_STATE` fixes all splits and estimator randomness.

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
print('pandas:', pd.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)

## 1. Load the supplied files

If the two CSV files are not already next to this notebook, the next cell opens Colab's upload dialog. Upload exactly the provided files; do not merge them.

In [ ]:
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'UNSW_NB15_training-set.csv'
TEST_PATH = DATA_DIR / 'UNSW_NB15_testing-set.csv'

if not (TRAIN_PATH.exists() and TEST_PATH.exists()):
    try:
        from google.colab import files
        print('Upload UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv')
        files.upload()
    except ImportError as exc:
        raise FileNotFoundError('Place both supplied CSV files beside the notebook.') from exc

assert TRAIN_PATH.exists() and TEST_PATH.exists(), 'Both supplied CSV files are required.'
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
print(f'Training file: {train_raw.shape[0]:,} rows × {train_raw.shape[1]} columns')
print(f'Testing file : {test_raw.shape[0]:,} rows × {test_raw.shape[1]} columns')
display(train_raw.head())

## 2. Data audit

The audit is deliberately performed before modelling. It checks types, missingness, non-finite numeric values, exact duplicate records, class balance, and the consistency of the two labels.

In [ ]:
def audit_frame(df, name):
    numeric = df.select_dtypes(include=np.number)
    summary = pd.Series({
        'rows': len(df),
        'columns': df.shape[1],
        'missing_cells': int(df.isna().sum().sum()),
        'infinite_numeric_cells': int(np.isinf(numeric.to_numpy()).sum()),
        'exact_duplicate_rows': int(df.duplicated().sum()),
        'attack_cat/label_inconsistencies': int(((df['attack_cat'] == 'Normal') != (df['label'] == 0)).sum())
    }, name=name)
    return summary

audit = pd.concat([audit_frame(train_raw, 'training'), audit_frame(test_raw, 'testing')], axis=1)
display(audit)
print('\nColumn data types (training):')
display(train_raw.dtypes.to_frame('dtype').T)
print('\nCategorical cardinality (training):')
display(train_raw[['proto', 'service', 'state']].nunique().to_frame('unique_values').T)
print('\nClass counts:')
display(pd.concat([train_raw['label'].value_counts().rename('training'),
                   test_raw['label'].value_counts().rename('testing')], axis=1).fillna(0).astype(int))
display(pd.concat([train_raw['attack_cat'].value_counts().rename('training'),
                   test_raw['attack_cat'].value_counts().rename('testing')], axis=1).fillna(0).astype(int))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
train_raw['label'].value_counts().sort_index().plot.bar(ax=axes[0], color=['#66c2a5', '#fc8d62'])
axes[0].set_title('Training distribution: binary label')
axes[0].set_xlabel('0 = Normal, 1 = Attack')
train_raw['attack_cat'].value_counts().sort_values().plot.barh(ax=axes[1], color=plt.cm.viridis(np.linspace(0.15, 0.9, 10)))
axes[1].set_title('Training distribution: attack category')
axes[1].set_xlabel('Number of flows')
plt.tight_layout()
plt.show()

print('Training class proportions for Experiment 1:')
display((train_raw['attack_cat'].value_counts(normalize=True) * 100).round(3).to_frame('percent'))

## 3. Feature policy and leakage-safe preprocessing

`id` is a record identifier rather than traffic behaviour, so it is removed. Both `attack_cat` and `label` are always removed from predictors: retaining either would leak the current target. We replace non-finite values defensively, but do not fit any statistics here. Imputers, encoders, and scaler are fitted only inside each training pipeline.

In [ ]:
TARGET_COLUMNS = ['attack_cat', 'label']
DROP_COLUMNS = ['id'] + TARGET_COLUMNS
CATEGORICAL_COLUMNS = ['proto', 'service', 'state']
FEATURE_COLUMNS = [c for c in train_raw.columns if c not in DROP_COLUMNS]
NUMERIC_COLUMNS = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_COLUMNS]

def clean_features(df):
    # Do not impute/scale here: that would risk fitting outside a pipeline.
    return df.loc[:, FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).copy()

X_all = clean_features(train_raw)
X_test_all = clean_features(test_raw)
assert list(X_all.columns) == list(X_test_all.columns)
print(f'Predictors: {len(FEATURE_COLUMNS)} = {len(NUMERIC_COLUMNS)} numeric + {len(CATEGORICAL_COLUMNS)} categorical')
print('Dropped:', DROP_COLUMNS)

def make_preprocessor():
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    return ColumnTransformer([
        ('numeric', numeric_pipe, NUMERIC_COLUMNS),
        ('categorical', categorical_pipe, CATEGORICAL_COLUMNS)
    ], remainder='drop')

def make_pipeline(model):
    return Pipeline([('preprocess', make_preprocessor()), ('model', model)])

## 4. Reusable selection and evaluation functions

A 90/10 stratified split of the supplied training file selects hyperparameters. Macro F1 is the selection measure because it gives a rare class the same importance as a common class. The final test file is passed only to `evaluate_final` after selection.

In [ ]:
MODEL_CANDIDATES = {
    'Logistic Regression': [{'C': 0.3}, {'C': 1.0}],
    'Random Forest': [{'max_depth': 20}, {'max_depth': None}]
}

def build_model(name, params=None, class_weight=None):
    params = params or {}
    if name == 'Logistic Regression':
        return LogisticRegression(C=params.get('C', 1.0), max_iter=1000, solver='lbfgs',
                                  class_weight=class_weight, random_state=RANDOM_STATE)
    if name == 'Random Forest':
        return RandomForestClassifier(n_estimators=params.get('n_estimators', 120),
                                      max_depth=params.get('max_depth', 20),
                                      min_samples_leaf=params.get('min_samples_leaf', 1),
                                      class_weight=class_weight, random_state=RANDOM_STATE,
                                      n_jobs=-1)
    raise ValueError(f'Unknown model: {name}')

def score_predictions(y_true, y_pred, probabilities, classes, binary=False):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    scores = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_macro, 'recall_macro': recall_macro, 'f1_macro': f1_macro,
        'precision_weighted': precision_weighted, 'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted
    }
    if binary:
        positive_index = list(classes).index(1)
        scores['roc_auc'] = roc_auc_score(y_true, probabilities[:, positive_index])
    else:
        scores['roc_auc_ovr_macro'] = roc_auc_score(y_true, probabilities, labels=classes,
                                                    multi_class='ovr', average='macro')
    return scores

def select_models(experiment, X, y, binary=False):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE)
    validation_rows, selected = [], {}
    for model_name, candidates in MODEL_CANDIDATES.items():
        for params in candidates:
            pipe = make_pipeline(build_model(model_name, params))
            pipe.fit(X_tr, y_tr)
            train_pred = pipe.predict(X_tr)
            val_pred = pipe.predict(X_val)
            val_prob = pipe.predict_proba(X_val)
            train_f1 = precision_recall_fscore_support(y_tr, train_pred, average='macro', zero_division=0)[2]
            val_scores = score_predictions(y_val, val_pred, val_prob, pipe.classes_, binary=binary)
            validation_rows.append({
                'experiment': experiment, 'model': model_name, 'parameters': json.dumps(params),
                'train_macro_f1': train_f1, 'validation_macro_f1': val_scores['f1_macro'],
                'validation_accuracy': val_scores['accuracy'],
                'validation_roc_auc': val_scores.get('roc_auc', val_scores.get('roc_auc_ovr_macro')),
                'overfit_gap_train_minus_validation': train_f1 - val_scores['f1_macro']
            })
        candidates_for_model = [r for r in validation_rows if r['model'] == model_name and r['experiment'] == experiment]
        selected[model_name] = max(candidates_for_model, key=lambda row: row['validation_macro_f1'])
    return pd.DataFrame(validation_rows), selected

def evaluate_final(experiment, model_name, pipe, X_holdout, y_holdout, labels, binary=False, setting='selected'):
    predicted = pipe.predict(X_holdout)
    probabilities = pipe.predict_proba(X_holdout)
    scores = score_predictions(y_holdout, predicted, probabilities, pipe.classes_, binary=binary)
    print(f'\n{experiment} | {model_name} | {setting}')
    print(classification_report(y_holdout, predicted, labels=labels, zero_division=0, digits=4))
    matrix = confusion_matrix(y_holdout, predicted, labels=labels)
    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.75), max(5, len(labels) * 0.65)))
    ConfusionMatrixDisplay(matrix, display_labels=labels).plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
    ax.set_title(f'{experiment}: {model_name} ({setting})')
    plt.tight_layout(); plt.show()
    return {'experiment': experiment, 'model': model_name, 'setting': setting, **scores}


## 5. Experiment 1 — Multi-class attack-category classification

Target: `attack_cat` (Normal and nine attack types). Each model gets a small validation search. The chosen setting is then refitted on **all** supplied training data and evaluated once on the supplied test data.

In [ ]:
y_multi = train_raw['attack_cat'].copy()
y_multi_test = test_raw['attack_cat'].copy()
multi_validation, multi_selected = select_models('Experiment 1: multiclass attack_cat', X_all, y_multi, binary=False)
display(multi_validation.sort_values(['model', 'validation_macro_f1'], ascending=[True, False]).style.format(precision=4))

multi_final_rows, multi_final_models = [], {}
multi_labels = sorted(y_multi.unique())
for model_name, choice in multi_selected.items():
    params = json.loads(choice['parameters'])
    final_pipe = make_pipeline(build_model(model_name, params)).fit(X_all, y_multi)
    multi_final_models[model_name] = final_pipe
    multi_final_rows.append(evaluate_final('Experiment 1: multiclass attack_cat', model_name, final_pipe,
                                           X_test_all, y_multi_test, multi_labels, binary=False,
                                           setting=f"selected {choice['parameters']}"))
multi_final = pd.DataFrame(multi_final_rows)
display(multi_final.style.format(precision=4))

## 6. Experiment 2 — Binary Normal versus Attack

Target: `label` (0 = Normal, 1 = Attack). Here ROC-AUC is the usual binary ROC-AUC based on the positive (Attack) probability.

In [ ]:
y_binary = train_raw['label'].copy()
y_binary_test = test_raw['label'].copy()
binary_validation, binary_selected = select_models('Experiment 2: Normal vs Attack', X_all, y_binary, binary=True)
display(binary_validation.sort_values(['model', 'validation_macro_f1'], ascending=[True, False]).style.format(precision=4))

binary_final_rows, binary_final_models = [], {}
for model_name, choice in binary_selected.items():
    params = json.loads(choice['parameters'])
    final_pipe = make_pipeline(build_model(model_name, params)).fit(X_all, y_binary)
    binary_final_models[model_name] = final_pipe
    binary_final_rows.append(evaluate_final('Experiment 2: Normal vs Attack', model_name, final_pipe,
                                            X_test_all, y_binary_test, [0, 1], binary=True,
                                            setting=f"selected {choice['parameters']}"))
binary_final = pd.DataFrame(binary_final_rows)
display(binary_final.style.format(precision=4))

## 7. Experiment 3 — Imbalance: Normal versus Worms

Worms is selected because it is the rarest category (130 training records). The two pre-specified treatments are identical except for `class_weight`: no weighting versus `balanced`. This is a cost-sensitive comparison, not a hyperparameter search on the test set. Report the Worms row, macro F1, and the false-positive/false-negative counts rather than only accuracy.

In [ ]:
rare_classes = ['Normal', 'Worms']
rare_train_mask = train_raw['attack_cat'].isin(rare_classes)
rare_test_mask = test_raw['attack_cat'].isin(rare_classes)
X_rare = X_all.loc[rare_train_mask].reset_index(drop=True)
X_rare_test = X_test_all.loc[rare_test_mask].reset_index(drop=True)
y_rare = (train_raw.loc[rare_train_mask, 'attack_cat'].eq('Worms')).astype(int).reset_index(drop=True)
y_rare_test = (test_raw.loc[rare_test_mask, 'attack_cat'].eq('Worms')).astype(int).reset_index(drop=True)
print('Normal/Worms training counts:', y_rare.value_counts().sort_index().to_dict())
print('Normal/Worms testing counts :', y_rare_test.value_counts().sort_index().to_dict())

# Fixed architecture: only class weighting changes, isolating the imbalance intervention.
rare_architecture = {'Logistic Regression': {'C': 1.0},
                     'Random Forest': {'n_estimators': 120, 'max_depth': 20}}
rare_final_rows = []
for model_name, params in rare_architecture.items():
    for setting, weight in [('unweighted', None), ('class_weight=balanced', 'balanced')]:
        rare_pipe = make_pipeline(build_model(model_name, params, class_weight=weight)).fit(X_rare, y_rare)
        rare_final_rows.append(evaluate_final('Experiment 3: Normal vs Worms', model_name, rare_pipe,
                                              X_rare_test, y_rare_test, [0, 1], binary=True, setting=setting))
rare_final = pd.DataFrame(rare_final_rows)
display(rare_final.style.format(precision=4))

# Explicit operational counts: TN, FP, FN, TP for the rare (Worms=1) class.
print('For interpretation: in each matrix, bottom-left = missed Worms (FN); top-right = normal false alarms (FP).')

## 8. Comparative tables and export

Use these tables in the report. A positive train-minus-validation gap is an overfitting warning. The final-test table is the only basis for final generalisation claims.

In [ ]:
tuning_results = pd.concat([multi_validation, binary_validation], ignore_index=True)
final_test_metrics = pd.concat([multi_final, binary_final, rare_final], ignore_index=True)

print('Validation search results')
display(tuning_results.sort_values(['experiment', 'model', 'validation_macro_f1'], ascending=[True, True, False]).style.format(precision=4))
print('\nFinal held-out test results')
display(final_test_metrics.style.format(precision=4))

tuning_results.to_csv('lab01_tuning_results.csv', index=False)
final_test_metrics.to_csv('lab01_final_test_metrics.csv', index=False)
print('Saved: lab01_tuning_results.csv and lab01_final_test_metrics.csv')

# Compact, auditable conclusion prompts based on measured—not assumed—results.
for experiment, frame in final_test_metrics.groupby('experiment'):
    best = frame.loc[frame['f1_macro'].idxmax()]
    print(f"{experiment}: highest final macro F1 = {best['f1_macro']:.4f} ({best['model']}, {best['setting']}).")

## Final conclusion checklist

When presenting, quote the exported final-test table and answer: (1) which model has the best macro F1 for multiclass attribution; (2) how binary detection differs from attribution; (3) whether weighting improved **Worms recall/macro F1** and what false-positive cost it caused. Do not use test metrics to make further tuning decisions.